In [ ]:
#!/usr/bin/env python3
"""
Fraktur OCR Implementation using Python
Supports multiple OCR engines with Fraktur-specific configurations
"""

import cv2
import numpy as np
from PIL import Image
import pytesseract
import argparse
import os
from pathlib import Path

class FrakturOCR:
    def __init__(self, engine='tesseract'):
        """
        Initialize Fraktur OCR processor

        Args:
            engine (str): OCR engine to use ('tesseract', 'easyocr')
        """
        self.engine = engine
        self.setup_engine()

    def setup_engine(self):
        """Setup the OCR engine with Fraktur-specific configurations"""
        if self.engine == 'tesseract':
            # Tesseract configuration for Fraktur
            # Using German language model and specific configs
            self.tesseract_config = r'--oem 3 --psm 6 -l deu_frak'

            # Alternative configs for different scenarios
            self.configs = {
                'default': r'--oem 3 --psm 6 -l deu_frak',
                'single_block': r'--oem 3 --psm 6 -l deu_frak',
                'single_line': r'--oem 3 --psm 7 -l deu_frak',
                'word': r'--oem 3 --psm 8 -l deu_frak',
                'sparse': r'--oem 3 --psm 11 -l deu_frak'
            }
        elif self.engine == 'easyocr':
            try:
                import easyocr
                self.reader = easyocr.Reader(['de'])
            except ImportError:
                raise ImportError("EasyOCR not installed. Install with: pip install easyocr")

    def preprocess_image(self, image_path, preprocessing_type='standard'):
        """
        Preprocess image for better Fraktur OCR results

        Args:
            image_path (str): Path to the input image
            preprocessing_type (str): Type of preprocessing to apply

        Returns:
            numpy.ndarray: Preprocessed image
        """
        # Load image
        if isinstance(image_path, str):
            img = cv2.imread(image_path)
        else:
            img = image_path

        if img is None:
            raise ValueError(f"Could not load image from {image_path}")

        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        if preprocessing_type == 'standard':
            # Standard preprocessing for Fraktur
            # Increase contrast
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)

            # Gaussian blur to reduce noise
            gray = cv2.GaussianBlur(gray, (1, 1), 0)

            # Threshold to binary
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        elif preprocessing_type == 'aggressive':
            # More aggressive preprocessing for poor quality images
            # Resize image (upscale for better recognition)
            height, width = gray.shape
            gray = cv2.resize(gray, (width*2, height*2), interpolation=cv2.INTER_CUBIC)

            # Denoise
            gray = cv2.fastNlMeansDenoising(gray)

            # Enhance contrast
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)

            # Morphological operations
            kernel = np.ones((1,1), np.uint8)
            gray = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)

            # Threshold
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        elif preprocessing_type == 'minimal':
            # Minimal preprocessing
            _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

        return binary

    def extract_text_tesseract(self, image, config_type='default'):
        """
        Extract text using Tesseract OCR

        Args:
            image (numpy.ndarray): Preprocessed image
            config_type (str): Configuration type to use

        Returns:
            str: Extracted text
        """
        config = self.configs.get(config_type, self.configs['default'])

        # Convert numpy array to PIL Image
        pil_image = Image.fromarray(image)

        # Extract text
        text = pytesseract.image_to_string(pil_image, config=config)

        return text.strip()

    def extract_text_easyocr(self, image):
        """
        Extract text using EasyOCR

        Args:
            image (numpy.ndarray): Preprocessed image

        Returns:
            str: Extracted text
        """
        results = self.reader.readtext(image)

        # Combine all detected text
        text_parts = []
        for (bbox, text, confidence) in results:
            if confidence > 0.5:  # Filter low confidence detections
                text_parts.append(text)

        return ' '.join(text_parts)

    def post_process_text(self, text):
        """
        Post-process extracted text for common Fraktur OCR errors

        Args:
            text (str): Raw OCR text

        Returns:
            str: Cleaned text
        """
        # Common Fraktur OCR corrections
        corrections = {
            'ſ': 's',  # Long s to regular s
            'ß': 'ss', # German eszett (optional)
            'ä': 'ae', # Umlauts (optional)
            'ö': 'oe',
            'ü': 'ue',
            'Ä': 'Ae',
            'Ö': 'Oe',
            'Ü': 'Ue',
        }

        # Apply corrections
        cleaned_text = text
        for old, new in corrections.items():
            cleaned_text = cleaned_text.replace(old, new)

        # Remove extra whitespace
        cleaned_text = ' '.join(cleaned_text.split())

        return cleaned_text

    def process_image(self, image_path, preprocessing='standard',
                     config_type='default', post_process=True):
        """
        Complete OCR processing pipeline

        Args:
            image_path (str): Path to input image
            preprocessing (str): Preprocessing type
            config_type (str): OCR configuration type
            post_process (bool): Whether to apply post-processing

        Returns:
            dict: Results including raw text, cleaned text, and confidence
        """
        # Preprocess image
        processed_image = self.preprocess_image(image_path, preprocessing)

        # Extract text based on engine
        if self.engine == 'tesseract':
            raw_text = self.extract_text_tesseract(processed_image, config_type)
        elif self.engine == 'easyocr':
            raw_text = self.extract_text_easyocr(processed_image)
        else:
            raise ValueError(f"Unsupported engine: {self.engine}")

        # Post-process if requested
        if post_process:
            cleaned_text = self.post_process_text(raw_text)
        else:
            cleaned_text = raw_text

        return {
            'raw_text': raw_text,
            'cleaned_text': cleaned_text,
            'preprocessing': preprocessing,
            'config': config_type if self.engine == 'tesseract' else 'default'
        }

    def batch_process(self, input_directory, output_directory=None,
                     file_pattern='*.jpg', **kwargs):
        """
        Process multiple images in batch

        Args:
            input_directory (str): Directory containing images
            output_directory (str): Directory to save results
            file_pattern (str): File pattern to match
            **kwargs: Additional arguments for process_image

        Returns:
            dict: Batch processing results
        """
        input_path = Path(input_directory)
        if output_directory:
            output_path = Path(output_directory)
            output_path.mkdir(parents=True, exist_ok=True)

        results = {}

        # Process all matching files
        for image_file in input_path.glob(file_pattern):
            try:
                print(f"Processing: {image_file.name}")

                # Process image
                result = self.process_image(str(image_file), **kwargs)
                results[image_file.name] = result

                # Save result if output directory specified
                if output_directory:
                    output_file = output_path / f"{image_file.stem}.txt"
                    with open(output_file, 'w', encoding='utf-8') as f:
                        f.write(result['cleaned_text'])

            except Exception as e:
                print(f"Error processing {image_file.name}: {e}")
                results[image_file.name] = {'error': str(e)}

        return results


def main():
    """Command line interface for Fraktur OCR"""
    parser = argparse.ArgumentParser(description='Fraktur OCR Tool')
    parser.add_argument('input', help='Input image file or directory')
    parser.add_argument('-o', '--output', help='Output directory for batch processing')
    parser.add_argument('-e', '--engine', choices=['tesseract', 'easyocr'],
                       default='tesseract', help='OCR engine to use')
    parser.add_argument('-p', '--preprocessing',
                       choices=['standard', 'aggressive', 'minimal'],
                       default='standard', help='Preprocessing type')
    parser.add_argument('-c', '--config',
                       choices=['default', 'single_block', 'single_line', 'word', 'sparse'],
                       default='default', help='Tesseract configuration')
    parser.add_argument('--no-postprocess', action='store_true',
                       help='Skip post-processing')
    parser.add_argument('--batch', action='store_true',
                       help='Process directory of images')

    args = parser.parse_args()

    # Initialize OCR processor
    ocr = FrakturOCR(engine=args.engine)

    try:
        if args.batch or os.path.isdir(args.input):
            # Batch processing
            results = ocr.batch_process(
                input_directory=args.input,
                output_directory=args.output,
                preprocessing=args.preprocessing,
                config_type=args.config,
                post_process=not args.no_postprocess
            )

            print(f"\nProcessed {len(results)} images")
            for filename, result in results.items():
                if 'error' in result:
                    print(f"❌ {filename}: {result['error']}")
                else:
                    print(f"✅ {filename}: {len(result['cleaned_text'])} characters")

        else:
            # Single image processing
            result = ocr.process_image(
                image_path=args.input,
                preprocessing=args.preprocessing,
                config_type=args.config,
                post_process=not args.no_postprocess
            )

            print("Raw OCR Text:")
            print("-" * 50)
            print(result['raw_text'])
            print("\nCleaned Text:")
            print("-" * 50)
            print(result['cleaned_text'])

            # Save to file if output specified
            if args.output:
                output_file = Path(args.output) / f"{Path(args.input).stem}.txt"
                output_file.parent.mkdir(parents=True, exist_ok=True)
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(result['cleaned_text'])
                print(f"\nResult saved to: {output_file}")

    except Exception as e:
        print(f"Error: {e}")
        return 1

    return 0


if __name__ == "__main__":
    exit(main())